In [48]:
import h5py
import numpy as np
from PIL import Image
import os
import re
from pathlib import Path

def safe_name(name):
    # 清理 Windows 不允许的路径字符
    return re.sub(r'[<>:"/\\|?*]', '_', name)[-100:]

def extract_axf_color(axf_file: Path | str, 
                      spectral_path='com.xrite.Materials/<U+4E1D+7EF8+7EFF>-AES-20260326-<U+65E0+6E05+6F06>/com.xrite.Resources/DiffuseModel/Color/Data',
                      output_dir: None | str | Path = None):
    with h5py.File(axf_file, 'r') as f:
        
        if spectral_path in f:
            data = f[spectral_path][:]
            print("光谱数据提取成功！")
            
            # 简单的加权转换 (模拟人眼对红绿蓝的敏感度)
            # 31个点通常对应 400nm-700nm
            b = np.mean(data[0:10])   # 蓝色波段
            g = np.mean(data[10:20])  # 绿色波段
            r = np.mean(data[20:31])  # 红色波段
            
            # 归一化并放大到 0-1 范围
            scale = 1.0 / max(r, g, b)
            final_rgb = (r * scale, g * scale, b * scale)
            
            if output_dir is None:
                print(f"--- Blender 颜色参数 ---")
                print(f"R: {final_rgb[0]:.4f}")
                print(f"G: {final_rgb[1]:.4f}")
                print(f"B: {final_rgb[2]:.4f}")
            else:
                output_path = Path(output_dir) / "extracted_color.txt"
                with open(output_path, "w" ) as f_out:
                    f_out.write(f"--- Blender 颜色参数 ---\n")
                    f_out.write(f"R: {final_rgb[0]:.4f}\n")
                    f_out.write(f"G: {final_rgb[1]:.4f}\n")
                    f_out.write(f"B: {final_rgb[2]:.4f}\n")
        else:
            print("未找到光谱数据，请检查路径。")

def extract_axf(axf_file: Path | str, size_threshold=512):
    file_path = Path(axf_file)
    output_dir = file_path.parent / file_path.stem
    output_dir.mkdir(exist_ok=True)

    with h5py.File(file_path, 'r') as f:
        def extract_core(name, obj):
            if isinstance(obj, h5py.Dataset) and obj.ndim >= 2:
                # 找出大于等于阈值的维度
                shape = obj.shape
                if shape[0] < 10 or shape[1] < 10:
                    return
                
                if any(dim >= size_threshold for dim in shape):
                    print(shape)
                    if len(shape) == 3:
                        # 检查通道位置（假设通道在最后）       
                        channels = shape[2]
                        if channels not in [1, 3, 4]:
                            print(f"跳过非图像多通道数据: {name} (Channels: {channels})")
                            return
                        
                    print(f"提取核心资产: {name}")
                    data = obj[:]
                    
                    # 归一化处理
                    # if data.dtype == np.float32:
                    d_min, d_max = data.min(), data.max()
                    if d_max > d_min:
                        data = (data - d_min) / (d_max - d_min)
                    data = (data * 255).astype(np.uint8)
                    
                    # 处理单通道 (1323, 1323, 1) -> (1323, 1323)
                    if data.ndim == 3 and data.shape[2] == 1:
                        data = data[:, :, 0]
                    print(data)
                    img = Image.fromarray(data)
                    img.save(os.path.join(output_dir, f"{safe_name(name)}.png"))
                    
        f.visititems(extract_core)
        extract_axf_color(axf_file, output_dir=output_dir)

In [49]:
from glob import glob
folder = Path(r"C:\Users\zhengyang\Downloads\聚复")
l = glob(str(folder / "*.axf"))
for axf_file in l:
    extract_axf(axf_file, size_threshold=479)

(360, 480, 31)
跳过非图像多通道数据: com.xrite.Materials/<U+4E1D+7EF8+7EFF>-mat-20260326-<U+65E0+6E05+6F06>/com.xrite.Pantora/pantora.edit/EditInProgress/InputResources/DiffuseModel/Color/Data (Channels: 31)
(360, 480, 3)
提取核心资产: com.xrite.Materials/<U+4E1D+7EF8+7EFF>-mat-20260326-<U+65E0+6E05+6F06>/com.xrite.Pantora/pantora.edit/EditInProgress/InputResources/DiffuseModel/Normal/Data
[[[ 21  24 254]
  [ 21  24 254]
  [ 21  23 254]
  ...
  [ 21   8 254]
  [ 21   5 254]
  [ 21   6 254]]

 [[ 21  24 254]
  [ 21  23 254]
  [ 21  23 254]
  ...
  [ 21   6 254]
  [ 21   5 254]
  [ 21   7 254]]

 [[ 21  23 254]
  [ 21  23 254]
  [ 21  23 254]
  ...
  [ 21   6 254]
  [ 21   5 254]
  [ 21   8 254]]

 ...

 [[ 21  30 254]
  [ 21  30 254]
  [ 21  30 254]
  ...
  [ 21  15 254]
  [ 21  15 254]
  [ 21  16 254]]

 [[ 21  29 254]
  [ 21  31 254]
  [ 21  31 254]
  ...
  [ 21  15 254]
  [ 21  15 254]
  [ 21  17 254]]

 [[ 21  30 254]
  [ 21  31 254]
  [ 21  30 254]
  ...
  [ 21  16 254]
  [ 21  17 254]
  [ 21  17 

In [44]:
import h5py

def print_structure(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name} (Shape: {obj.shape}, Dtype: {obj.dtype})")

with h5py.File(r'C:\Users\zhengyang\Downloads\聚复\丝绸绿-mat-20260326-无清漆.axf', 'r') as f:
    f.visititems(print_structure)

Dataset: com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/BluePrimary (Shape: (2,), Dtype: float32)
Dataset: com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/GreenPrimary (Shape: (2,), Dtype: float32)
Dataset: com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/RedPrimary (Shape: (2,), Dtype: float32)
Dataset: com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/WhitePoint (Shape: (2,), Dtype: float32)
Dataset: com.xrite.ColorSpecs/sRGB,D65/com.xrite.ColorMatrices/XYZ (Shape: (3, 3), Dtype: float32)
Dataset: com.xrite.ColorSpecs/sRGB,D65/com.xrite.TristimulusSpec/Observer (Shape: (), Dtype: |S16)
Dataset: com.xrite.ColorSpecs/spectral/Type (Shape: (), Dtype: |S9)
Dataset: com.xrite.ColorSpecs/spectral/Version (Shape: (3,), Dtype: int32)
Dataset: com.xrite.ColorSpecs/spectral/Wavelengths (Shape: (31,), Dtype: float32)
Dataset: com.xrite.DeviceSpecs/X-Rite.MA-T12.4001142/Family (Shape: (), Dtype: |S6)
Dataset: com.xrite.DeviceSpecs/X-Rite.MA-T12.4001142/Manufacturer (Sha

In [63]:
import h5py
import numpy as np
from PIL import Image
import os
from pathlib import Path

def extract_pbr_assets(axf_file, size_threshold=200):
    f_path = Path(axf_file)
    # 创建以 PBR 属性命名的子文件夹
    out_dir = f_path.parent / f_path.stem
    out_dir.mkdir(exist_ok=True)
    
    with h5py.File(f_path, 'r') as f:
        # 1. 定位到 Resources 根目录
        # MA-T12 的核心 PBR 数据通常在这里

        def process_node(name, obj):
            if not isinstance(obj, h5py.Dataset):
                return

            root_path = "com.xrite.Resources"
            if root_path not in name:
                print(f"错误: 在 {name} 中找不到 Resources 根路径")
                return
            full_name = f"{root_path}/{name}"
            # 过滤掉不需要的 Measurements 原始测量数据
            if "Measurements" in full_name:
                return

            data_path = full_name.lower()
            shape = obj.shape
            
            # --- 逻辑 A: 提取纹理图 (H, W, C) ---
            if obj.ndim >= 2 and any(d >= size_threshold for d in shape):
                print(f"发现贴图资产: {full_name} {shape}")
                data = obj[:]
                
                # 全局缩放处理 (保持 RGB 比例，防止颜色丢失)
                if data.dtype != np.uint8:
                    d_max = data.max()
                    if d_max > 0:
                        # 映射到 0-255，但不改变通道间比例
                        data = (np.clip(data, 0, None) / d_max * 255).astype(np.uint8)
                
                # 处理多通道预览 (如 31 通道转 RGB 预览)
                if data.ndim == 3:
                    if data.shape[2] == 31: # 光谱纹理
                        # 取 550nm 附近的绿光波段作为参考预览，或者取前三层
                        data = data[:, :, 10:13] # 粗略取中段作为 RGB
                    elif data.shape[0] <= 4: # (C, H, W) 转 (H, W, C)
                        data = data.transpose(1, 2, 0)
                
                # 根据语义命名保存
                tag = "Unknown"
                if "color" in data_path: tag = "BaseColor"
                elif "normal" in data_path: tag = "Normal"
                elif "height" in data_path: tag = "Height"
                elif "lobes" in data_path: tag = "Roughness"
                
                img = Image.fromarray(data)
                img.save(out_dir / f"{tag}_{shape[0]}x{shape[1]}.png")

            # --- 逻辑 B: 提取物理数值 (如 Fresnel 或单点颜色) ---
            elif obj.ndim == 1 and (shape[0] == 31 or shape[0] == 3):
                print(f"发现数值资产: {full_name} {shape}")
                val_data = obj[:]
                with open(out_dir / "pbr_metadata.txt", "a", encoding="utf-8") as meta:
                    meta.write(f"{full_name}: {list(val_data)}\n")

        # 只遍历 Resources 文件夹
        f.visititems(process_node)

print("--- PBR 资产定向提取完成 ---")

--- PBR 资产定向提取完成 ---


In [64]:
from glob import glob
folder = Path(r"C:\Users\zhengyang\Downloads\聚复")
l = glob(str(folder / "*.axf"))
for axf_file in l:
    extract_pbr_assets(axf_file, size_threshold=479)

错误: 在 com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/BluePrimary 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/GreenPrimary 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/RedPrimary 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/sRGB,D65/com.xrite.Chromaticities/WhitePoint 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/sRGB,D65/com.xrite.ColorMatrices/XYZ 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/sRGB,D65/com.xrite.TristimulusSpec/Observer 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/spectral/Type 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/spectral/Version 中找不到 Resources 根路径
错误: 在 com.xrite.ColorSpecs/spectral/Wavelengths 中找不到 Resources 根路径
错误: 在 com.xrite.DeviceSpecs/X-Rite.MA-T12.4001142/Family 中找不到 Resources 根路径
错误: 在 com.xrite.DeviceSpecs/X-Rite.MA-T12.4001142/Manufacturer 中找不到 Resources 根路径
错误: 在 com.xrite.DeviceSpecs/X-Rite.MA-T12.4001142/Model 中找不到 Resources 根路径
错误: 在 com.xrite.DeviceSpecs/X-Rite.MA

TypeError: Cannot handle this data type: (1, 1, 1), |u1